# Stage 6c(a): leakage-status inventory (reviewer Point 3)

Companion notebook for `scripts/6c_ood_leakage_check.py`. That script builds a systematic,
code-verified inventory classifying every (training-target, evaluation-set) combination shown
in Figs 3-7 and SI Fig 6 as `IN-OBJECTIVE` (the eval set was part of what that model's outer
loop directly optimized against - not a generalization test), `OOD` (genuinely held out),
`BASELINE` (baseline emulator on its own training set, explicitly framed as a lower bound, not
a leakage claim), or `TRAJECTORY` (an NRMSE-vs-iteration curve on the model's own target - by
construction in-objective, and already framed that way in the manuscript).

This is documentation only - no new compute. Every classification is derived directly from the
code (`build_group_emis_dicts`/`generate_eval_data` in `utils_inverse.py`, and the exact
`train_scenarios`/`training_paths` lists each `load_fig*_data` function uses) and the
manuscript text - see each row's `basis` column for exactly what was checked.

In [1]:
import os, sys
PROJECT_ROOT = os.getcwd()
sys.path.insert(0, PROJECT_ROOT)

import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 200)

df = pd.read_csv(os.path.join(PROJECT_ROOT, "data", "SI_results", "leakage_inventory", "leakage_inventory.csv"))
print(f"{len(df)} entries across {df['figure'].nunique()} figures")
df["status"].value_counts()

163 entries across 6 figures


status
OOD                                                                               89
IN-OBJECTIVE                                                                      30
IN-OBJECTIVE (disclosed)                                                          28
BASELINE                                                                           7
TRAJECTORY                                                                         6
IN-OBJECTIVE (HEADLINE - contradicts manuscript's 'out-of-distribution' claim)     2
SUBTLE/INDIRECT IN-OBJECTIVE - not addressed by the manuscript's leakage claim     1
Name: count, dtype: int64

### Headline finding 1 - Figure 6's R$^2$=0.97 result is entirely in-objective

In [2]:
fig6 = df[df["figure"] == "Fig 6"]
fig6[["train_target", "eval_target", "status"]]

,train_target,eval_target,status
152,Opt. Tier 1,"DAMIP (M-GHG, M-aer)",OOD
153,Opt. Tier 1,GeoMIP (G6sulfur),OOD
154,Opt. DAMIP,"DAMIP (M-GHG, M-aer)",IN-OBJECTIVE
155,Opt. DAMIP,GeoMIP (G6sulfur),OOD
156,Opt. GeoMIP,GeoMIP (G6sulfur),IN-OBJECTIVE
157,Opt. GeoMIP,"DAMIP (M-GHG, M-aer)",OOD
158,Opt. All,"DAMIP (M-GHG, M-aer)",IN-OBJECTIVE (HEADLINE - contradicts manuscript's 'out-of-distribution' claim)
159,Opt. All,GeoMIP (G6sulfur),IN-OBJECTIVE (HEADLINE - contradicts manuscript's 'out-of-distribution' claim)


`Opt. All`'s row is the one presented in the manuscript text as the headline result
("...achieves high accuracy when evaluated on the out-of-distribution isolated forcing and
climate intervention subsets; emulating DAMIP and GeoMIP yields R$^2$=0.97"). Per
`scripts/3b_inverse_all_agents.py:129`, the `All` optimization target for the multi-agent case
is built with `DAMIP=True, GeoMIP=True` - so this specific evaluation is in-objective, not
out-of-distribution as stated. Every other cell in this table (Tier 1 vs. DAMIP/GeoMIP,
DAMIP vs. GeoMIP and vice versa) genuinely is OOD, and would still support a (smaller,
less dramatic) generalization claim.

### Headline finding 2 - Figure 7/MESM's leakage claim doesn't cover the upstream SCM channel

In [3]:
fig7 = df[df["figure"] == "Fig 7"]
fig7[["train_target", "eval_target", "status", "basis"]]

,train_target,eval_target,status,basis
160,"MESM baseline (own, Priority 1)",Priority 1 (own),BASELINE,'the baseline emulator inherently retains the highest skill on its own training data (Priority 1...
161,MESM-on-SCM-optimized-CO2 (constant/sine),"Priority 2 / DECK (direct, MESM's own loop)",OOD,MESM's own training loop never optimizes against its Priority 2/DECK eval sets - confirmed no su...
162,MESM-on-SCM-optimized-CO2 (constant/sine),"Priority 2 / DECK (indirect, via upstream SCM optimization)",SUBTLE/INDIRECT IN-OBJECTIVE - not addressed by the manuscript's leakage claim,"checkpoints/co2/inverse_{constant,sine}_all_co2_only_MESM.pkl use group='all', i.e. the SCM-side..."


MESM's own training loop genuinely never touches its Priority 2/DECK evaluation data - the
manuscript's "prevent any information leakage during training" claim is accurate in that direct
sense. But the CO$_2$ trajectory MESM trains on comes from an SCM-side optimization run against
`group='all'` (`checkpoints/co2/inverse_{constant,sine}_all_co2_only_MESM.pkl`), which itself
targeted the same Priority 2/DECK scenarios MESM is later evaluated against. This is a subtler,
second-order leakage channel the manuscript's framing doesn't address.

### Figure 4 / SI Fig 6 - the grid-structured leakage (disclosed by the manuscript)

In [4]:
grid = df[df["figure"].isin(["Fig 4", "SI Fig 6"])]
pd.crosstab(
    [grid["figure"], grid["panel"], grid["train_target"]],
    grid["eval_target"],
    values=grid["status"], aggfunc="first"
).fillna("")

eval_target                                                 CS3                      DECK                    Tier 1                    Tier 2
figure   panel           train_target                                                                                                        
Fig 4    (a) CO2-only    Opt. All      IN-OBJECTIVE (disclosed)  IN-OBJECTIVE (disclosed)  IN-OBJECTIVE (disclosed)  IN-OBJECTIVE (disclosed)
                         Opt. CS3                  IN-OBJECTIVE                       OOD                       OOD                       OOD
                         Opt. DECK                          OOD              IN-OBJECTIVE                       OOD                       OOD
                         Opt. Tier 1                        OOD                       OOD              IN-OBJECTIVE                       OOD
                         Opt. Tier 2                        OOD                       OOD                       OOD              IN-OBJECTIVE
         (b) Multi-agent Opt. All      IN-OBJECTIVE (disclosed)  IN-OBJECTIVE (disclosed)  IN-OBJECTIVE (disclosed)  IN-OBJECTIVE (disclosed)
                         Opt. CS3                  IN-OBJECTIVE                       OOD                       OOD                       OOD
                         Opt. DECK                          OOD              IN-OBJECTIVE                       OOD                       OOD
                         Opt. Tier 1                        OOD                       OOD              IN-OBJECTIVE                       OOD
                         Opt. Tier 2                        OOD                       OOD                       OOD              IN-OBJECTIVE
SI Fig 6 BC-only         Opt. All      IN-OBJECTIVE (disclosed)  IN-OBJECTIVE (disclosed)  IN-OBJECTIVE (disclosed)  IN-OBJECTIVE (disclosed)
                         Opt. CS3                  IN-OBJECTIVE                       OOD                       OOD                       OOD
                         Opt. DECK                          OOD              IN-OBJECTIVE                       OOD                       OOD
                         Opt. Tier 1                        OOD                       OOD              IN-OBJECTIVE                       OOD
                         Opt. Tier 2                        OOD                       OOD                       OOD              IN-OBJECTIVE
         CH4-only        Opt. All      IN-OBJECTIVE (disclosed)  IN-OBJECTIVE (disclosed)  IN-OBJECTIVE (disclosed)  IN-OBJECTIVE (disclosed)
                         Opt. CS3                  IN-OBJECTIVE                       OOD                       OOD                       OOD
                         Opt. DECK                          OOD              IN-OBJECTIVE                       OOD                       OOD
                         Opt. Tier 1                        OOD                       OOD              IN-OBJECTIVE                       OOD
                         Opt. Tier 2                        OOD                       OOD                       OOD              IN-OBJECTIVE
         CO2-only        Opt. All      IN-OBJECTIVE (disclosed)  IN-OBJECTIVE (disclosed)  IN-OBJECTIVE (disclosed)  IN-OBJECTIVE (disclosed)
                         Opt. CS3                  IN-OBJECTIVE                       OOD                       OOD                       OOD
                         Opt. DECK                          OOD              IN-OBJECTIVE                       OOD                       OOD
                         Opt. Tier 1                        OOD                       OOD              IN-OBJECTIVE                       OOD
                         Opt. Tier 2                        OOD                       OOD                       OOD              IN-OBJECTIVE
         N2O-only        Opt. All      IN-OBJECTIVE (disclosed)  IN-OBJECTIVE (disclosed)  IN-OBJECTIVE (disclosed)  IN-OBJECTIVE (disclosed)
                        

Each `Opt. X` row is in-objective exactly on its own column `X` (the diagonal-style pattern
below) and OOD everywhere else - this structure is visible directly from the axis labels, so
not a hidden problem. `Opt. All` is in-objective on every column, which the manuscript already
states explicitly ("optimizing over all scenario sets at once inherently introduces information
leakage", main text, immediately after Table 1).

### Figures 3 / 5 - trajectory plots (correctly framed, included for completeness)

In [5]:
traj = df[df["figure"].isin(["Fig 3", "Fig 5"])]
traj[["figure", "panel", "train_target", "eval_target", "status"]]

,figure,panel,train_target,eval_target,status
0,Fig 3,(a) CO2-only,Tier 1 (own),Tier 1 (own),TRAJECTORY
1,Fig 3,(b) CH4-only,Tier 1 (own),Tier 1 (own),TRAJECTORY
2,Fig 3,(c) N2O-only,Tier 1 (own),Tier 1 (own),TRAJECTORY
3,Fig 3,(d) Sulfur-only,Tier 1 (own),Tier 1 (own),TRAJECTORY
4,Fig 3,(e) BC-only,Tier 1 (own),Tier 1 (own),TRAJECTORY
5,Fig 3,(a) CO2-only,baseline (own),Tier 1 (own),BASELINE
6,Fig 3,(b) CH4-only,baseline (own),Tier 1 (own),BASELINE
7,Fig 3,(c) N2O-only,baseline (own),Tier 1 (own),BASELINE
8,Fig 3,(d) Sulfur-only,baseline (own),Tier 1 (own),BASELINE
9,Fig 3,(e) BC-only,baseline (own),Tier 1 (own),BASELINE


### Full inventory

In [6]:
df

,figure,panel,train_target,eval_target,status,basis
0,Fig 3,(a) CO2-only,Tier 1 (own),Tier 1 (own),TRAJECTORY,load_fig3_single_forcing_data: checkpoints/{a}/inverse_constant_tier1_{a}_only.pkl 'errors' arra...
1,Fig 3,(b) CH4-only,Tier 1 (own),Tier 1 (own),TRAJECTORY,load_fig3_single_forcing_data: checkpoints/{a}/inverse_constant_tier1_{a}_only.pkl 'errors' arra...
2,Fig 3,(c) N2O-only,Tier 1 (own),Tier 1 (own),TRAJECTORY,load_fig3_single_forcing_data: checkpoints/{a}/inverse_constant_tier1_{a}_only.pkl 'errors' arra...
3,Fig 3,(d) Sulfur-only,Tier 1 (own),Tier 1 (own),TRAJECTORY,load_fig3_single_forcing_data: checkpoints/{a}/inverse_constant_tier1_{a}_only.pkl 'errors' arra...
4,Fig 3,(e) BC-only,Tier 1 (own),Tier 1 (own),TRAJECTORY,load_fig3_single_forcing_data: checkpoints/{a}/inverse_constant_tier1_{a}_only.pkl 'errors' arra...
5,Fig 3,(a) CO2-only,baseline (own),Tier 1 (own),BASELINE,"baseline_{a}_only.pkl['Tier 1']['mean'] - baseline evaluated on its own training set, explicitly..."
6,Fig 3,(b) CH4-only,baseline (own),Tier 1 (own),BASELINE,"baseline_{a}_only.pkl['Tier 1']['mean'] - baseline evaluated on its own training set, explicitly..."
7,Fig 3,(c) N2O-only,baseline (own),Tier 1 (own),BASELINE,"baseline_{a}_only.pkl['Tier 1']['mean'] - baseline evaluated on its own training set, explicitly..."
8,Fig 3,(d) Sulfur-only,baseline (own),Tier 1 (own),BASELINE,"baseline_{a}_only.pkl['Tier 1']['mean'] - baseline evaluated on its own training set, explicitly..."
9,Fig 3,(e) BC-only,baseline (own),Tier 1 (own),BASELINE,"baseline_{a}_only.pkl['Tier 1']['mean'] - baseline evaluated on its own training set, explicitly..."
